In [1]:
import networkx as nx
import numpy as np
from scipy.io import mmread
import pandas as pd


from sklearn import metrics

import copy
import scipy.sparse as sp
import os
import anndata as ad
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import scanpy as sc
import scirpy as ir
import pandas as pd
import awkward as ak

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)



/home/phile/miniconda3/envs/mvTCR/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from mvtcr.utils_preprocessing import Preprocessing


/home/phile/miniconda3/envs/mvTCR/lib/python3.10/site-packages/comet_ml/env_logging.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
# train, val = Preprocessing.stratified_group_shuffle_split(gene_TCR, stratify_col='donor', group_col='clonotype', test_size=0.2, random_seed=42)

# gene_TCR.obs['set'] = 'train'
# gene_TCR.obs.loc[val, 'set'] = 'val'

# gene_TCR.obs["set"].value_counts()

In [4]:
gene_TCR = sc.read_h5ad('./data/gene_TCR_mvTCR_ready_for_running_integration.h5ad')

In [5]:
gene_TCR.obs = gene_TCR.obs.reset_index(drop=False)
gene_TCR.obs

,index,v_gene_TRA,v_gene_TRB,d_gene_TRA,d_gene_TRB,j_gene_TRA,j_gene_TRB,c_gene_TRA,c_gene_TRB,cdr3_TRA,...,n_genes,mt_fraction,receptor_type,receptor_subtype,chain_pairing,clonotype,clonotype_size,alpha_len,beta_len,set
0,AGGGTGAGTATTACCG-18_0,TRAV19,TRBV20-1,None,TRBD2,TRAJ40,TRBJ2-3,TRAC,TRBC2,CALSEASGTYKYIF,...,1830,0.058050,TCR,TRA+TRB,single pair,0,3166,11,13,val
1,CTTGGCTTCGTTGCCT-25_1,TRAV23DV6,TRBV7-2,None,TRBD1,TRAJ48,TRBJ2-5,TRAC,TRBC2,CAAILFGNEKLTF,...,1382,0.063250,TCR,TRA+TRB,single pair,1,5666,13,14,train
2,ACGATACTCGCAGGCT-40_2,TRAV9-2,TRBV7-6,None,TRBD2,TRAJ49,TRBJ2-3,TRAC,TRBC2,CALSADTGNQFYF,...,1279,0.067611,TCR,TRA+TRB,single pair,2,125,15,14,train
3,ACGCCAGTCATGTCTT-8_3,TRAV12-3,TRBV20-1,None,TRBD1,TRAJ32,TRBJ2-7,TRAC,TRBC2,CAMNPAWGGATNKLIF,...,1669,0.054716,TCR,TRA+TRB,single pair,3,230,11,16,train
4,TTCTTAGCAAAGAATC-4_4,TRAV13-2,TRBV4-1,None,TRBD2,TRAJ37,TRBJ2-1,TRAC,TRBC2,CAETALGNTGKLIF,...,2683,0.044035,TCR,TRA+TRB,single pair,4,1,15,13,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145474,GAAGCAGAGCAGGCTA-3_145474,TRAV22,TRBV24-1,None,TRBD2,TRAJ47,TRBJ2-2,TRAC,TRBC2,CAVEPLYGNKLVF,...,789,0.135747,TCR,TRA+TRB,single pair,58159,1,14,13,train
145475,CAGTCCTTCATCACCC-8_145475,TRAV29DV5,TRBV7-6,None,TRBD2,TRAJ50,TRBJ2-3,TRAC,TRBC2,CAATHASGYDKVIF,...,1468,0.041688,TCR,TRA+TRB,single pair,1286,14,15,16,train
145476,GACTACACACGGTAAG-3_145476,TRAV21,TRBV6-6,None,TRBD2,TRAJ50,TRBJ2-3,TRAC,TRBC2,CAVDLMKTSYDKVIF,...,1646,0.047574,TCR,TRA+TRB,single pair,58160,1,16,15,train
145477,ATCGAGTAGTTTCCTT-4_145477,TRAV3,TRBV5-1,None,TRBD2,TRAJ34,TRBJ2-2,TRAC,TRBC2,CAVREGRGTDKLIF,...,782,0.137519,TCR,TRA+TRB,single pair,4623,22,14,14,train


In [6]:
ls_index = os.listdir('./5 to 1 Indices')
ls_index

['smart_aligned_v2_train_indices_random_split_2.npy',
 'smart_aligned_v2_train_indices_random_split_4.npy',
 'smart_aligned_v2_train_indices_tcr_split_4.npy',
 'smart_aligned_v2_test_indices_tcr_ab_split_1.npy',
 'smart_aligned_v2_test_indices_random_split_2.npy',
 'smart_aligned_v2_train_indices_tcr_split_3.npy',
 'smart_aligned_v2_train_indices_tcr_ab_split_5.npy',
 'smart_aligned_v2_val_indices_tcr_split_4.npy',
 'smart_aligned_v2_train_indices_tcr_split_1.npy',
 'smart_aligned_v2_test_indices_random_split_3.npy',
 'smart_aligned_v2_test_indices_tcr_ab_split_3.npy',
 'smart_aligned_v2_val_indices_tcr_split_5.npy',
 'smart_aligned_v2_val_indices_tcr_ab_split_1.npy',
 'smart_aligned_v2_train_indices_random_split_1.npy',
 'smart_aligned_v2_val_indices_random_split_1.npy',
 'smart_aligned_v2_val_indices_random_split_3.npy',
 'smart_aligned_v2_test_indices_tcr_split_3.npy',
 'smart_aligned_v2_val_indices_tcr_ab_split_4.npy',
 'smart_aligned_v2_val_indices_tcr_ab_split_5.npy',
 'smart_ali

In [7]:
ref_dat = pd.read_csv('./data/smart_aligned_v2_dataset_reference_5_to_1.csv')
ref_dat

,tcr,peptide,label,tcr_source_dataset,tcr_source_index,peptide_source_dataset,peptide_source_index,donor,binding_tcr
0,CASSLYEQYF,GILGFVFTL,1,10X,0,10X,0,donor1,Y
1,CAWTGTGKIGWDSPLHF,KLGGALQAK,1,10X,2,10X,2,donor1,Y
2,CASSWGGGSHYGYTF,IVTDFSVIK,1,10X,3,10X,3,donor1,Y
3,CASSLYSATGELFF,AVFDRKSDAK,1,10X,5,10X,5,donor1,Y
4,CASSLYSATGELFF,AVFDRKSDAK,1,10X,6,10X,6,donor1,Y
...,...,...,...,...,...,...,...,...,...
431041,CASSFGRGEGEQYF,ELAGIGILTV,0,10X,39940,10X,258,NaN,N
431042,CASSPHFQVDTGELFF,IVTDFSVIK,0,10X,85427,10X,3,NaN,Y
431043,CASSLMRGGTYNSPLHF,FLYALALLL,0,10X,72515,10X,134,NaN,N
431044,CASSVSSTDTQYF,GILGFVFTL,0,10X,133931,10X,0,NaN,Y


In [11]:
from mvtcr.models.model_selection import run_model_selection
timeout = (400*60)
n_samples = 3
n_gpus = 1
seed = 42

In [9]:
import io
from contextlib import redirect_stdout

In [15]:
for i in ls_index[1:]:
    if i.endswith('.npy'):
        data_ix = np.load(os.path.join('./5 to 1 Indices', i), allow_pickle=True)
        ref_dat_index = ref_dat.iloc[data_ix]
        gene_TCR_integration = gene_TCR[ref_dat_index['tcr_source_index'].values].copy()
        params_experiment = {
            'study_name': '10X_mvTCR_embedding_cluster',
            'comet_workspace': None, 
            'model_name': 'moe',
            'balanced_sampling': 'clonotype',
            'metadata': [],
            'save_path': f'./mvTCR_integration_res_5_to_1/saved_models/{i.split(".")[0]}/10X_mvTCR_embedding_cluster',
            'conditional': 'donor_ohe',
            'n_epochs': 6,
        }
        params_optimization = {
            'name': 'knn_prediction',
            'prediction_column': 'antigen',
        }
        f = io.StringIO()
        with redirect_stdout(f):
            run_model_selection(gene_TCR_integration, params_experiment, params_optimization, n_samples, timeout, n_gpus, sampler_seed=seed)


            captured_output = f.getvalue()
            best_model = captured_output.splitlines()[5].strip(" ")

            import mvtcr.utils_training as utils


            path_model = f'./mvTCR_integration_res_5_to_1/saved_models/{i.split(".")[0]}/10X_mvTCR_embedding_cluster/{best_model}/best_model_by_reconstruction.pt'
            model = utils.load_model(gene_TCR_integration, path_model)


            mvTCR_integration_latent = model.get_latent(gene_TCR_integration, metadata=[], return_mean=False).X
            mvTCR_integration_latent.shape
            pd.DataFrame(mvTCR_integration_latent).to_csv(f"./mvTCR_integration_res_5_to_1/10X_mvTCR_embedding_integration_latent_best_reconstruction_metric_{i.split('.')[0]}.csv", index=False)


[I 2026-04-03 14:26:14,219] A new study created in RDB with name: 10X_mvTCR_embedding_cluster
100%|██████████| 6/6 [10:14<00:00, 102.48s/it]
[I 2026-04-03 14:36:35,102] Trial 0 finished with value: 0.5688531618389147 and parameters: {'dropout': 0.1, 'activation': 'linear', 'rna_hidden': 1500, 'hdim': 200, 'shared_hidden': 100, 'rna_num_layers': 1, 'tfmr_encoding_layers': 4, 'loss_weights_kl': 4.0428727350273357e-07, 'loss_weights_tcr': 0.034702669886504146, 'lr': 1.0994335574766187e-05, 'zdim': 50, 'tfmr_embedding_size': 16, 'tfmr_num_heads': 8, 'tfmr_dropout': 0.15000000000000002}. Best is trial 0 with value: 0.5688531618389147.
100%|██████████| 6/6 [04:45<00:00, 47.59s/it]
[I 2026-04-03 14:41:27,283] Trial 1 finished with value: 0.5519191632756539 and parameters: {'dropout': 0.1, 'activation': 'linear', 'rna_hidden': 1000, 'hdim': 300, 'shared_hidden': 300, 'rna_num_layers': 3, 'tfmr_encoding_layers': 1, 'loss_weights_kl': 1.2173252504194046e-07, 'loss_weights_tcr': 0.009163741808778

In [ ]:
# path_model = f'./mvTCR_integration_res_5_to_1/saved_models/{i.split(".")[0]}/10X_mvTCR_embedding_cluster/{best_model}/best_model_by_reconstruction.pt'
# model = utils.load_model(gene_TCR_integration, path_model)
# mvTCR_integration_latent = model.get_latent(gene_TCR[ref_dat_index['tcr_source_index'].values], metadata=[], return_mean=False).X
# mvTCR_integration_latent.shape
# pd.DataFrame(mvTCR_integration_latent).to_csv(f"./mvTCR_integration_res_5_to_1/10X_mvTCR_embedding_integration_latent_best_reconstruction_metric_{i.split('.')[0]}.csv", index=False)

In [ ]:
import io
from contextlib import redirect_stdout

In [16]:
for i in ls_index:
    if i.endswith('.npy'):

        mvTCR_integration_latent = pd.read_csv(f"./mvTCR_integration_res_5_to_1/10X_mvTCR_embedding_integration_latent_best_reconstruction_metric_{i.split('.')[0]}.csv")
        print(i.split('.')[0])
        print(mvTCR_integration_latent.shape)
        

smart_aligned_v2_train_indices_random_split_2
(301732, 10)
smart_aligned_v2_train_indices_random_split_4
(301732, 10)
smart_aligned_v2_train_indices_tcr_split_4
(283032, 10)
smart_aligned_v2_test_indices_tcr_ab_split_1
(79602, 50)
smart_aligned_v2_test_indices_random_split_2
(64657, 10)
smart_aligned_v2_train_indices_tcr_split_3
(275780, 50)
smart_aligned_v2_train_indices_tcr_ab_split_5
(277048, 50)
smart_aligned_v2_val_indices_tcr_split_4
(70758, 50)
smart_aligned_v2_train_indices_tcr_split_1
(278515, 10)
smart_aligned_v2_test_indices_random_split_3
(64657, 10)
smart_aligned_v2_test_indices_tcr_ab_split_3
(87232, 50)
smart_aligned_v2_val_indices_tcr_split_5
(70433, 50)
smart_aligned_v2_val_indices_tcr_ab_split_1
(70289, 10)
smart_aligned_v2_train_indices_random_split_1
(301732, 50)
smart_aligned_v2_val_indices_random_split_1
(64657, 50)
smart_aligned_v2_val_indices_random_split_3
(64657, 50)
smart_aligned_v2_test_indices_tcr_split_3
(86321, 50)
smart_aligned_v2_val_indices_tcr_ab_spli

In [ ]:
import mvtcr.utils_training as utils

In [ ]:
utils.load_model(gene_TCR_integration, path_model)

In [ ]:
for i in ls_index[1:2]:
    if i.endswith('.npy'): 
        data_ix = np.load(os.path.join('./5 to 1 Indices', i), allow_pickle=True)
        ref_dat_index = ref_dat.iloc[data_ix]
        gene_TCR_integration = gene_TCR[ref_dat_index['tcr_source_index'].values].copy()
        for best_model in ['trial_0']:           
            path_model = f'./mvTCR_integration_res_5_to_1/saved_models/{i.split(".")[0]}/10X_mvTCR_embedding_cluster/{best_model}/best_model_by_reconstruction.pt'
            model = utils.load_model(gene_TCR_integration, path_model)
            print(best_model)
            print(i)
            mvTCR_integration_latent = model.get_latent(gene_TCR_integration, metadata=[], return_mean=False).X
            print(mvTCR_integration_latent.shape)

In [ ]:
# for i in ls_index[3:4]:
#     if i.endswith('.npy'): 
#         data_ix = np.load(os.path.join('./5 to 1 Indices', i), allow_pickle=True)
#         ref_dat_index = ref_dat.iloc[data_ix]
#         gene_TCR_integration = gene_TCR[ref_dat_index['tcr_source_index'].values].copy()
#         for best_model in ['trial_0', 'trial_1','trial_2', 'trial_3','trial_4']:           
#             path_model = f'./mvTCR_integration_res_5_to_1/saved_models/{i.split(".")[0]}/10X_mvTCR_embedding_cluster/{best_model}/best_model_by_reconstruction.pt'
#             model = utils.load_model(gene_TCR_integration, path_model)
#             print(best_model)
#             print(i)
#             mvTCR_integration_latent = model.get_latent(gene_TCR_integration, metadata=[], return_mean=False).X
#             print(mvTCR_integration_latent.shape)

In [18]:
for i in ls_index:
    if i.endswith('.npy'): 
        data_ix = np.load(os.path.join('./5 to 1 Indices', i), allow_pickle=True)
        ref_dat_index = ref_dat.iloc[data_ix]
        gene_TCR_integration = gene_TCR[ref_dat_index['tcr_source_index'].values].copy()
        for best_model in ['trial_0']:           
            path_model = f'./mvTCR_integration_res/saved_models/{i.split(".")[0]}/10X_mvTCR_embedding_cluster/{best_model}/best_model_by_reconstruction.pt'
            model = utils.load_model(gene_TCR_integration, path_model)
            print(best_model)
            print(i)
            mvTCR_integration_latent = model.get_latent(gene_TCR_integration, metadata=[], return_mean=False).X
            print(mvTCR_integration_latent.shape)
            pd.DataFrame(mvTCR_integration_latent).to_csv(f"./mvTCR_integration_res_5_to_1_new/10X_mvTCR_embedding_integration_latent_best_reconstruction_metric_{i.split('.')[0]}.csv", index=False)


trial_0
smart_aligned_v2_train_indices_random_split_2.npy
(301732, 50)
trial_0
smart_aligned_v2_train_indices_random_split_4.npy
(301732, 50)
trial_0
smart_aligned_v2_train_indices_tcr_split_4.npy
(283032, 50)
trial_0
smart_aligned_v2_test_indices_tcr_ab_split_1.npy
(79602, 50)
trial_0
smart_aligned_v2_test_indices_random_split_2.npy
(64657, 50)
trial_0
smart_aligned_v2_train_indices_tcr_split_3.npy
(275780, 50)
trial_0
smart_aligned_v2_train_indices_tcr_ab_split_5.npy
(277048, 50)
trial_0
smart_aligned_v2_val_indices_tcr_split_4.npy
(70758, 50)
trial_0
smart_aligned_v2_train_indices_tcr_split_1.npy
(278515, 50)
trial_0
smart_aligned_v2_test_indices_random_split_3.npy
(64657, 50)
trial_0
smart_aligned_v2_test_indices_tcr_ab_split_3.npy
(87232, 50)
trial_0
smart_aligned_v2_val_indices_tcr_split_5.npy
(70433, 50)
trial_0
smart_aligned_v2_val_indices_tcr_ab_split_1.npy
(70289, 50)
trial_0
smart_aligned_v2_train_indices_random_split_1.npy
(301732, 50)
trial_0
smart_aligned_v2_val_indices_r

In [ ]:
for i in ls_index:
    if i.endswith('.npy') and ("indices_tcr_split_5" in i): 
        data_ix = np.load(os.path.join('./3 to 1 Indices', i), allow_pickle=True)
        ref_dat_index = ref_dat.iloc[data_ix]
        gene_TCR_integration = gene_TCR[ref_dat_index['tcr_source_index'].values].copy()
        for best_model in ['trial_4']:           
            path_model = f'./mvTCR_integration_res/saved_models/{i.split(".")[0]}/10X_mvTCR_embedding_cluster/{best_model}/best_model_by_reconstruction.pt'
            model = utils.load_model(gene_TCR_integration, path_model)
            print(best_model)
            print(i)
            mvTCR_integration_latent = model.get_latent(gene_TCR_integration, metadata=[], return_mean=False).X
            print(mvTCR_integration_latent.shape)
            print(data_ix.shape)

In [20]:
ls_index = os.listdir('./5 to 1 Indices')
ls_index
for i in ls_index:
    if i.endswith('.npy'): 
        data_ix = np.load(os.path.join('./5 to 1 Indices', i), allow_pickle=True)
        ref_dat_index = ref_dat.iloc[data_ix]
        # gene_TCR_integration = gene_TCR[ref_dat_index['tcr_source_index'].values].copy()
        check = pd.read_csv(f"./mvTCR_integration_res_5_to_1_new/10X_mvTCR_embedding_integration_latent_best_reconstruction_metric_{i.split('.')[0]}.csv")
        print(check.shape[0] == data_ix.shape[0])
        print(i)
        # print()

True
smart_aligned_v2_train_indices_random_split_2.npy
True
smart_aligned_v2_train_indices_random_split_4.npy
True
smart_aligned_v2_train_indices_tcr_split_4.npy
True
smart_aligned_v2_test_indices_tcr_ab_split_1.npy
True
smart_aligned_v2_test_indices_random_split_2.npy
True
smart_aligned_v2_train_indices_tcr_split_3.npy
True
smart_aligned_v2_train_indices_tcr_ab_split_5.npy
True
smart_aligned_v2_val_indices_tcr_split_4.npy
True
smart_aligned_v2_train_indices_tcr_split_1.npy
True
smart_aligned_v2_test_indices_random_split_3.npy
True
smart_aligned_v2_test_indices_tcr_ab_split_3.npy
True
smart_aligned_v2_val_indices_tcr_split_5.npy
True
smart_aligned_v2_val_indices_tcr_ab_split_1.npy
True
smart_aligned_v2_train_indices_random_split_1.npy
True
smart_aligned_v2_val_indices_random_split_1.npy
True
smart_aligned_v2_val_indices_random_split_3.npy
True
smart_aligned_v2_test_indices_tcr_split_3.npy
True
smart_aligned_v2_val_indices_tcr_ab_split_4.npy
True
smart_aligned_v2_val_indices_tcr_ab_spli

10X_mvTCR_embedding_integration_latent_best_reconstruction_metric_smart_aligned_v2_train_indices_tcr_split_5
Should be (191806, 45)

In [ ]:
gene_TCR[ref_dat_index['tcr_source_index'].values]